In [1]:
%pip install psycopg[binary]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg

In [3]:
from psycopg import sql

In [5]:
%pip install random

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement random (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for random


In [4]:
import random
import string
import pandas as pd


def generate_site_code():
    letters = "".join(random.choices(string.ascii_uppercase, k=3))
    numbers = "".join(random.choices(string.digits, k=3))
    return letters + numbers


def generate_site_data(count):
    data = []
    used_site_codes = set()
    used_coordinates = set()

    while len(data) < count:

        site_code = generate_site_code()
        latitude = round(random.uniform(-90, 90), 2)
        longitude = round(random.uniform(-180, 180), 2)

        # Skip duplicate site codes
        if site_code in used_site_codes:
            continue

        # Skip duplicate coordinates
        if (latitude, longitude) in used_coordinates:
            continue

        used_site_codes.add(site_code)
        used_coordinates.add((latitude, longitude))

        data.append(
            {
                "site_code": site_code,
                "latitude": latitude,
                "longitude": longitude,
            }
        )

    return pd.DataFrame(data)

In [5]:
def create_db_meta(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [6]:
create_db_meta("meta")

Database 'meta' created successfully!


In [8]:
def create_table_meta():
    try:
        with psycopg.connect(
            dbname="meta",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS metadata (
                        site_name VARCHAR(100) Not NULL,
                        latitude DOUBLE PRECISION Not NULL,
                        longitude DOUBLE PRECISION Not NULL
                    );
                """)

            conn.commit()
            print("metadata table created.")

    except psycopg.Error as e:
        print(e)

In [9]:
create_table_meta()

metadata table created.


In [10]:
def insert_sites():

    sites_df = generate_site_data(10000)

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:
        with conn.cursor() as cur:
            for _, row in sites_df.iterrows():
                cur.execute(
                    """
                    INSERT INTO metadata
                    (site_name, latitude, longitude)
                    VALUES (%s, %s, %s)                                                                         
                    """,
                    (row["site_code"], row["latitude"], row["longitude"]),
                )
        conn.commit()

    print("Sites inserted successfully!")

In [11]:
insert_sites()

Sites inserted successfully!


In [12]:
def get_sites():

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute(""" 
                SELECT site_name, latitude, longitude
                FROM metadata
            """)

            sites = cur.fetchall()

    return sites

In [ ]:
print(type(get_sites()))

In [13]:
data = list(get_sites())

In [14]:
type(data[0])

tuple

In [15]:
len(data)

10000

In [16]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
def create_db_site_weather(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [19]:
create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [ ]:
import time
import datetime
import requests
import psycopg

# ============================================================
# CONFIGURATION
# ============================================================

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

DB_NAME = "site_weather"

API_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

BATCH_SIZE = 30

# Wait after a successful request
REQUEST_DELAY = 0.5

# Wait before retrying a failed request
RETRY_WAIT = 5

# Wait when API rate limit (429) happens
RATE_LIMIT_WAIT = 60

REQUEST_TIMEOUT = 120


# ============================================================
# CREATE TABLE
# ============================================================

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS site_weather (
                site_name VARCHAR(100) NOT NULL,
                time_interval TIMESTAMPTZ NOT NULL,
                temperature REAL NOT NULL,
                humidity REAL NOT NULL,
                solar_radiance REAL NOT NULL,

                UNIQUE (site_name, time_interval)
            );
        """)

    conn.commit()


# ============================================================
# FIND COMPLETED SITES
# ============================================================

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT site_name
            FROM site_weather
            GROUP BY site_name
            HAVING COUNT(*) = 24;
        """)

        completed_sites = {row[0] for row in cur.fetchall()}


# ============================================================
# BUILD PENDING SITE LIST
# ============================================================

pending_sites = []

for row in data:

    site_name = row[0]
    latitude = float(row[1])
    longitude = float(row[2])

    if site_name not in completed_sites:
        pending_sites.append((site_name, latitude, longitude))


print("=" * 60)
print("RESUME CHECK")
print("=" * 60)
print(f"Total sites       : {len(data)}")
print(f"Completed sites   : {len(completed_sites)}")
print(f"Remaining sites   : {len(pending_sites)}")
print("=" * 60)


# ============================================================
# FETCH ONE BATCH
# ============================================================


def fetch_batch(batch):

    site_names = [site[0] for site in batch]
    latitudes = [str(site[1]) for site in batch]
    longitudes = [str(site[2]) for site in batch]

    params = {
        "latitude": ",".join(latitudes),
        "longitude": ",".join(longitudes),
        "hourly": ",".join(
            [
                "temperature_2m",
                "relative_humidity_2m",
                "direct_radiation",
            ]
        ),
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    # Keep trying the SAME batch until it succeeds.
    while True:

        try:

            print()
            print(f"Requesting {len(batch)} sites...")
            print(f"{site_names[0]} -> {site_names[-1]}")

            response = requests.get(
                API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            if response.status_code == 200:

                result = response.json()

                if isinstance(result, dict):
                    result = [result]

                if not isinstance(result, list):
                    raise ValueError("Invalid API response format.")

                if len(result) != len(batch):
                    raise ValueError(
                        f"Expected {len(batch)} locations, "
                        f"but API returned {len(result)}."
                    )

                print("API SUCCESS")
                return result

            # ------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------

            if response.status_code == 429:

                print(
                    f"API rate limit reached. " f"Waiting {RATE_LIMIT_WAIT} seconds..."
                )

                time.sleep(RATE_LIMIT_WAIT)

                print("Retrying SAME batch...")
                continue

            # ------------------------------------------------
            # SERVER ERROR
            # ------------------------------------------------

            if response.status_code >= 500:

                print(
                    f"Server error {response.status_code}. "
                    f"Waiting {RETRY_WAIT} seconds..."
                )

                time.sleep(RETRY_WAIT)

                print("Retrying SAME batch...")
                continue

            # ------------------------------------------------
            # OTHER API ERROR
            # ------------------------------------------------

            print(
                f"API error {response.status_code}. " f"Waiting {RETRY_WAIT} seconds..."
            )

            time.sleep(RETRY_WAIT)

            print("Retrying SAME batch...")

        # ----------------------------------------------------
        # NETWORK / REQUEST ERROR
        # ----------------------------------------------------

        except (requests.Timeout, requests.ConnectionError) as e:

            print(f"Network error: {e}")
            print(f"Waiting {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)
            print("Retrying SAME batch...")

        # ----------------------------------------------------
        # JSON / OTHER REQUEST ERROR
        # ----------------------------------------------------

        except (requests.RequestException, ValueError) as e:

            print(f"Request/response error: {e}")
            print(f"Waiting {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)
            print("Retrying SAME batch...")


# ============================================================
# EXTRACT ONE SITE
# ============================================================


def extract_site_weather(site_name, weather):

    hourly = weather.get("hourly")

    if not hourly:
        raise ValueError(f"No hourly data for {site_name}")

    times = hourly.get("time", [])
    temperatures = hourly.get("temperature_2m", [])
    humidity = hourly.get("relative_humidity_2m", [])
    radiation = hourly.get("direct_radiation", [])

    # We need exactly 24 records for one day.
    if not (
        len(times) == 24
        and len(temperatures) == 24
        and len(humidity) == 24
        and len(radiation) == 24
    ):
        raise ValueError(
            f"Invalid data for {site_name}: "
            f"time={len(times)}, "
            f"temperature={len(temperatures)}, "
            f"humidity={len(humidity)}, "
            f"radiation={len(radiation)}"
        )

    rows = []

    for i in range(24):

        timestamp = datetime.datetime.fromisoformat(times[i]).replace(
            tzinfo=datetime.timezone.utc
        )

        rows.append(
            (
                site_name,
                timestamp,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            )
        )

    return rows


# ============================================================
# SAVE BATCH
# ============================================================


def save_batch(cursor, batch, api_results):

    all_rows = []

    for i, site in enumerate(batch):

        site_name = site[0]
        weather = api_results[i]

        rows = extract_site_weather(
            site_name,
            weather,
        )

        all_rows.extend(rows)

    cursor.executemany(
        """
        INSERT INTO site_weather (
            site_name,
            time_interval,
            temperature,
            humidity,
            solar_radiance
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (site_name, time_interval)
        DO NOTHING;
    """,
        all_rows,
    )

    return len(all_rows)


# ============================================================
# PROCESS ALL BATCHES
# ============================================================

total_pending = len(pending_sites)

if total_pending == 0:

    print("All sites are already complete.")

else:

    total_batches = (total_pending + BATCH_SIZE - 1) // BATCH_SIZE    

    with psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cur:

            for batch_number, start in enumerate(
                range(0, total_pending, BATCH_SIZE),
                start=1,
            ):

                batch = pending_sites[start : start + BATCH_SIZE]

                print()
                print("=" * 60)
                print(f"BATCH {batch_number}/{total_batches}")
                print(f"Sites: {start + 1} - " f"{start + len(batch)}")
                print("=" * 60)

                # --------------------------------------------
                # API
                # --------------------------------------------

                api_results = fetch_batch(batch)

                # --------------------------------------------
                # DATABASE
                # --------------------------------------------

                rows_inserted = save_batch(
                    cur,
                    batch,
                    api_results,
                )

                # --------------------------------------------
                # SAVE IMMEDIATELY
                # --------------------------------------------

                conn.commit()

                total_rows += rows_inserted

                print(
                    f"Batch complete | "
                    f"Sites: {len(batch)} | "
                    f"Rows: {rows_inserted} | "
                    f"Total rows: {total_rows}"
                )

                time.sleep(REQUEST_DELAY)


# ============================================================
# FINAL VERIFICATION
# ============================================================

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT COUNT(DISTINCT site_name)
            FROM site_weather;
        """)

        actual_sites = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM site_weather;
        """)

        actual_rows = cur.fetchone()[0]


expected_sites = len(data)
expected_rows = expected_sites * 24

print()
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)
print(f"Expected sites : {expected_sites}")
print(f"Actual sites   : {actual_sites}")
print(f"Expected rows  : {expected_rows}")
print(f"Actual rows    : {actual_rows}")
print("=" * 60)

if actual_sites == expected_sites and actual_rows == expected_rows:
    print("SUCCESS: All sites and all hourly records are complete.")
else:
    print("NOT COMPLETE YET.")
    print(f"Missing rows: {expected_rows - actual_rows}")

In [ ]:
import time
import datetime
import requests
import psycopg

# ============================================================
# CONFIGURATION
# ============================================================

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

# Maximum coordinates/sites in one API request
BATCH_SIZE = 30

# Delay between successful API requests
REQUEST_DELAY = 0.5

# Used only when API does not provide Retry-After
MINUTE_WAIT = 60
HOUR_WAIT = 3600

# Network/server retry backoff
MAX_BACKOFF = 300

REQUEST_TIMEOUT = 120


# ============================================================
# CREATE WEATHER TABLE
# ============================================================

with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS site_weather (
                site_name VARCHAR(100) NOT NULL,
                time_interval TIMESTAMPTZ NOT NULL,
                temperature REAL NOT NULL,
                humidity REAL NOT NULL,
                solar_radiance REAL NOT NULL,

                UNIQUE (
                    site_name,
                    time_interval
                )
            );
            """)

    conn.commit()


# ============================================================
# FIND COMPLETED SITES
# ============================================================

with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT site_name
            FROM site_weather
            GROUP BY site_name
            HAVING COUNT(*) = 24;
            """)

        completed_sites = {row[0] for row in cur.fetchall()}


print("=" * 70)

print("RESUME CHECK")

print("=" * 70)

print(f"Sites in meta       : {len(data)}")

print(f"Already completed   : {len(completed_sites)}")

print(f"Remaining sites     : " f"{len(data) - len(completed_sites)}")

print("=" * 70)


# ============================================================
# BUILD PENDING SITE LIST
# ============================================================

pending_sites = []

for row in data:

    site_name = row[0]
    latitude = float(row[1])
    longitude = float(row[2])

    if site_name not in completed_sites:

        pending_sites.append((site_name, latitude, longitude))


# ============================================================
# RETRY-AFTER PARSER
# ============================================================


def get_retry_after(response):

    value = response.headers.get("Retry-After")

    if value is None:
        return None

    # Retry-After can be seconds
    try:

        seconds = float(value)

        if seconds >= 0:
            return int(seconds)

    except (ValueError, TypeError):

        pass

    # Retry-After can also be an HTTP date
    try:

        from email.utils import parsedate_to_datetime

        retry_date = parsedate_to_datetime(value)

        if retry_date.tzinfo is None:

            retry_date = retry_date.replace(tzinfo=datetime.timezone.utc)

        now = datetime.datetime.now(datetime.timezone.utc)

        seconds = (retry_date - now).total_seconds()

        return max(0, int(seconds))

    except Exception:

        return None


# ============================================================
# DETECT RATE LIMIT
# ============================================================


def detect_rate_limit(response):

    text = response.text.lower()

    headers_text = " ".join(str(value).lower() for value in response.headers.values())

    combined = text + " " + headers_text

    # Hourly limit
    hourly_words = [
        "hourly",
        "per hour",
        "hour limit",
        "hourly limit",
        "hourly quota",
        "quota per hour",
    ]

    for word in hourly_words:

        if word in combined:

            return "hour"

    # Minute limit
    minute_words = [
        "minute",
        "per minute",
        "minute limit",
        "minute quota",
        "requests/min",
    ]

    for word in minute_words:

        if word in combined:

            return "minute"

    return None


# ============================================================
# CALCULATE WAIT TIME
# ============================================================


def get_rate_limit_wait(response):

    # --------------------------------------------------------
    # FIRST PRIORITY: Retry-After
    # --------------------------------------------------------

    retry_after = get_retry_after(response)

    if retry_after is not None:

        return (retry_after, "Retry-After")

    # --------------------------------------------------------
    # SECOND PRIORITY: Detect limit type
    # --------------------------------------------------------

    limit_type = detect_rate_limit(response)

    if limit_type == "hour":

        return (HOUR_WAIT, "hourly limit")

    if limit_type == "minute":

        return (MINUTE_WAIT, "minute limit")

    # --------------------------------------------------------
    # UNKNOWN 429
    #
    # We do NOT continuously hit the API every 60 seconds.
    # Safe fallback is one hour.
    # --------------------------------------------------------

    return (HOUR_WAIT, "unknown 429")


# ============================================================
# WAIT WITH COUNTDOWN
# ============================================================


def wait_with_countdown(seconds):

    remaining = int(seconds)

    while remaining > 0:

        minutes = remaining // 60
        secs = remaining % 60

        print(f"\rWaiting " f"{minutes:02d}:{secs:02d}", end="", flush=True)

        sleep_time = min(10, remaining)

        time.sleep(sleep_time)

        remaining -= sleep_time

    print("\rWait complete.          ")


# ============================================================
# FETCH ONE BATCH
# ============================================================


def fetch_batch(batch):

    site_names = [site[0] for site in batch]

    latitudes = [str(site[1]) for site in batch]

    longitudes = [str(site[2]) for site in batch]

    params = {
        "latitude": ",".join(latitudes),
        "longitude": ",".join(longitudes),
        "hourly": ",".join(
            ["temperature_2m", "relative_humidity_2m", "direct_radiation"]
        ),
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    temporary_retry_count = 0

    while True:

        try:

            print("\n" + "-" * 70)

            print(f"API REQUEST")

            print(f"Batch size: {len(batch)}")

            print(f"Sites: " f"{site_names[0]} " f"-> " f"{site_names[-1]}")

            print("-" * 70)

            response = requests.get(
                OPEN_METEO_URL, params=params, timeout=REQUEST_TIMEOUT
            )

            status = response.status_code

            # =================================================
            # SUCCESS
            # =================================================

            if status == 200:

                result = response.json()

                # Multiple locations normally return a list.
                # For one site, API may return one dictionary.
                if isinstance(result, dict):

                    result = [result]

                if not isinstance(result, list):

                    raise RuntimeError("Unexpected API response format.")

                if len(result) != len(batch):

                    raise RuntimeError(
                        f"API returned "
                        f"{len(result)} locations "
                        f"but requested "
                        f"{len(batch)}."
                    )

                print(f"API SUCCESS " f"({len(result)} sites)")

                return result

            # =================================================
            # 429 RATE LIMIT
            # =================================================

            if status == 429:

                wait_seconds, reason = get_rate_limit_wait(response)

                print("\n" + "=" * 70)

                print("API RATE LIMIT REACHED")

                print(f"Reason       : {reason}")

                print(f"Retry-After  : " f"{response.headers.get('Retry-After')}")

                print(f"Waiting      : " f"{wait_seconds} seconds")

                print("=" * 70)

                wait_with_countdown(wait_seconds)

                # Retry the SAME batch
                temporary_retry_count = 0

                continue

            # =================================================
            # 408 REQUEST TIMEOUT
            # =================================================

            if status == 408:

                temporary_retry_count += 1

                wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

                print(f"HTTP 408. " f"Retrying in " f"{wait_seconds}s...")

                time.sleep(wait_seconds)

                continue

            # =================================================
            # 5XX SERVER ERRORS
            # =================================================

            if 500 <= status <= 599:

                temporary_retry_count += 1

                wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

                print(f"HTTP {status}. " f"Retrying in " f"{wait_seconds}s...")

                time.sleep(wait_seconds)

                continue

            # =================================================
            # OTHER 4XX
            # =================================================

            if 400 <= status <= 499:

                raise RuntimeError(
                    f"Permanent API error " f"HTTP {status}: " f"{response.text[:1000]}"
                )

            # =================================================
            # UNKNOWN HTTP STATUS
            # =================================================

            temporary_retry_count += 1

            wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

            print(f"Unexpected HTTP {status}. " f"Retrying in " f"{wait_seconds}s...")

            time.sleep(wait_seconds)

        # =====================================================
        # NETWORK TIMEOUT
        # =====================================================

        except requests.Timeout as e:

            temporary_retry_count += 1

            wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

            print(f"Network timeout: {e}")

            print(f"Retrying in " f"{wait_seconds}s...")

            time.sleep(wait_seconds)

        # =====================================================
        # CONNECTION ERROR
        # =====================================================

        except requests.ConnectionError as e:

            temporary_retry_count += 1

            wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

            print(f"Connection error: {e}")

            print(f"Retrying in " f"{wait_seconds}s...")

            time.sleep(wait_seconds)

        # =====================================================
        # OTHER REQUEST ERROR
        # =====================================================

        except requests.RequestException as e:

            temporary_retry_count += 1

            wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

            print(f"Request error: {e}")

            print(f"Retrying in " f"{wait_seconds}s...")

            time.sleep(wait_seconds)


# ============================================================
# VALIDATE ONE SITE'S API RESULT
# ============================================================


def extract_site_weather(site_name, weather):

    hourly = weather.get("hourly")

    if hourly is None:

        raise RuntimeError(f"No hourly data for " f"{site_name}")

    times = hourly.get("time", [])

    temperatures = hourly.get("temperature_2m", [])

    humidity = hourly.get("relative_humidity_2m", [])

    radiation = hourly.get("direct_radiation", [])

    # --------------------------------------------------------
    # We expect exactly 24 hourly records
    # --------------------------------------------------------

    if not (
        len(times) == 24
        and len(temperatures) == 24
        and len(humidity) == 24
        and len(radiation) == 24
    ):

        raise RuntimeError(
            f"Invalid data for {site_name}: "
            f"time={len(times)}, "
            f"temperature={len(temperatures)}, "
            f"humidity={len(humidity)}, "
            f"radiation={len(radiation)}"
        )

    rows = []

    for i in range(24):

        timestamp = datetime.datetime.fromisoformat(times[i]).replace(
            tzinfo=datetime.timezone.utc
        )

        rows.append(
            (
                site_name,
                timestamp,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            )
        )

    return rows


# ============================================================
# SAVE BATCH
# ============================================================


def save_batch(cursor, batch, api_results):

    all_rows = []

    for i, site in enumerate(batch):

        site_name = site[0]

        weather = api_results[i]

        site_rows = extract_site_weather(site_name, weather)

        all_rows.extend(site_rows)

    # ========================================================
    # BULK INSERT
    # ========================================================

    cursor.executemany(
        """
        INSERT INTO site_weather (
            site_name,
            time_interval,
            temperature,
            humidity,
            solar_radiance
        )
        VALUES (
            %s,
            %s,
            %s,
            %s,
            %s
        )
        ON CONFLICT (
            site_name,
            time_interval
        )
        DO NOTHING;
        """,
        all_rows,
    )

    return len(all_rows)


# ============================================================
# PROCESS BATCHES
# ============================================================

total_pending = len(pending_sites)


if total_pending == 0:

    print("\nAll 10,000 sites are " "already complete.")

else:

    total_batches = (total_pending + BATCH_SIZE - 1) // BATCH_SIZE                                                                                                                                    

    total_rows_this_run = 0

    with psycopg.connect(
        dbname="site_weather",
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cur:

            for batch_number, start in enumerate(
                range(0, total_pending, BATCH_SIZE), start=1
            ):

                batch = pending_sites[start : start + BATCH_SIZE]

                print("\n" + "=" * 70)

                print(f"BATCH " f"{batch_number}/" f"{total_batches}")

                print(f"Pending sites: " f"{start + 1}" f"-" f"{start + len(batch)}")

                print("=" * 70)

                # ------------------------------------------------
                # API
                # ------------------------------------------------

                api_results = fetch_batch(batch)

                # ------------------------------------------------
                # DATABASE
                # ------------------------------------------------

                rows_inserted = save_batch(cur, batch, api_results)

                # ------------------------------------------------
                # COMMIT IMMEDIATELY
                # ------------------------------------------------

                conn.commit()

                total_rows_this_run += rows_inserted

                print(f"\nBATCH SUCCESS")

                print(f"Sites completed: " f"{len(batch)}")

                print(f"Rows written: " f"{rows_inserted}")

                print(f"Rows this run: " f"{total_rows_this_run}")

                # ------------------------------------------------
                # Small delay
                # ------------------------------------------------

                time.sleep(REQUEST_DELAY)


# ============================================================
# FINAL DATABASE VERIFICATION
# ============================================================

with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT COUNT(DISTINCT site_name)
            FROM site_weather;
            """)

        distinct_sites = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM site_weather;
            """)

        total_rows = cur.fetchone()[0]


print("\n" + "=" * 70)

print("FINAL VERIFICATION")

print("=" * 70)

print(f"Expected sites : {len(data)}")

print(f"Actual sites   : {distinct_sites}")

print(f"Expected rows  : " f"{len(data) * 24}")

print(f"Actual rows    : {total_rows}")

print("=" * 70)


if distinct_sites == len(data) and total_rows == len(data) * 24:

    print("SUCCESS!")

    print("All 10,000 sites have " "24 hourly weather records.")

else:

    print("NOT COMPLETE YET.")

    print(f"Missing rows: " f"{(len(data) * 24) - total_rows}")

RESUME CHECK
Sites in meta       : 10000
Already completed   : 0
Remaining sites     : 10000

BATCH 1/334
Pending sites: 1-30

----------------------------------------------------------------------
API REQUEST
Batch size: 30
Sites: LCI211 -> IWS132
----------------------------------------------------------------------
API SUCCESS (30 sites)

BATCH SUCCESS
Sites completed: 30
Rows written: 720
Rows this run: 720

BATCH 2/334
Pending sites: 31-60

----------------------------------------------------------------------
API REQUEST
Batch size: 30
Sites: KGW061 -> EVC840
----------------------------------------------------------------------
API SUCCESS (30 sites)

BATCH SUCCESS
Sites completed: 30
Rows written: 720
Rows this run: 1440

BATCH 3/334
Pending sites: 61-90

----------------------------------------------------------------------
API REQUEST
Batch size: 30
Sites: OWB425 -> YJR878
----------------------------------------------------------------------
API SUCCESS (30 sites)

BATCH SUC